# B6.2 — Test evaluation, multi-label datasets: OmniCount-191 test copy and PASCAL VOC 2007 test

Pre-declared in `results/B6_0_Setup_and_Rules/B6_rule_file.json`.

| Method | Mode | Datasets |
|---|---|---|
| GPV_R4+P[MixA] (primary), GPV_K-n+P[LM0] (secondary) — frozen `candidates_v2` | multi-label, U_both; **every class of the vocabulary queried** (standard protocol); the present-classes protocol is a subset of the same predictions | OmniCount-191 test (1,957 images, domain vocabularies), VOC 2007 test (4,952 images, 20 classes) |

**Order:** OmniCount-191 first (~12 h), then VOC 2007 (~20 h, estimate). **Resumable** — Run all again after a disconnect or to continue in a later session.
Set `RUN` in Step 2.1 to run one dataset per session if you prefer. Check Step 1.1 (sanity check) before leaving it overnight.

In [1]:
# Step 0.1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 0.2 — Paths, installs (SAN + detectron2 + diffusers, protobuf last), repo
# ============================================================
import importlib, json, re, shutil, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np, pandas as pd
import torch
assert torch.cuda.is_available(), "Use a GPU runtime (L4)."
PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
B50 = RESULTS_ROOT / "B5_0_OmniCount_Region_Tests"
OUT_DIR = RESULTS_ROOT / "B6_2_Ours_Multi"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OMNICOUNT_ROOT = PROJECT_ROOT / "datasets" / "OmniCount-191"
FSC147_IMAGE_DIR = PROJECT_ROOT / "datasets" / "FSC147" / "raw" / "images_384_VarV2"
DATASETS = PROJECT_ROOT / "datasets"
FSC147_RAW = DATASETS / "FSC147" / "raw"
VOC_TAR = DATASETS / "VOCtest_06-Nov-2007.tar"
OMNICOUNT_ROOT = DATASETS / "OmniCount-191"
LOCAL = Path("/content/data"); LOCAL.mkdir(parents=True, exist_ok=True)
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME, "| L4 expected for exact live checks")

DETECTRON2_COMMIT = "1e3e13bbf607b54f62205c4c33922521822fb298"
%cd /content
if not Path("/content/SAN").exists():
    !git clone -q https://github.com/MendelXu/SAN.git /content/SAN
!pip install -q cython scipy shapely timm h5py submitit scikit-image wandb setuptools fvcore tabulate regex opencv-python "open_clip_torch==2.16.0"
!pip uninstall -y -q detectron2
!python -m pip install -q --no-build-isolation "git+https://github.com/facebookresearch/detectron2.git@{DETECTRON2_COMMIT}"
!pip install -q "accelerate==1.14.0" "diffusers==0.40.0" "transformers==5.16.1"
!pip install -q "protobuf==3.20.3" "iopath==0.1.10"

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
for p in [str(REPO_DIR), "/content/SAN"]:
    if p not in sys.path:
        sys.path.insert(0, p)
print("Repo:", REPO_COMMIT[:10])

GPU: NVIDIA L4 | L4 expected for exact live checks
/content
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
tensorboard 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 3.20.3 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.21.0 requires protobuf>=4.25.2; python_v

In [3]:
# ============================================================
# Step 0.3 — SAM2 (frozen commit), module imports, models
# ============================================================
import google.protobuf, importlib.metadata as md_meta
assert md_meta.version("protobuf") == "3.20.3" and google.protobuf.__version__ == "3.20.3", \
    "protobuf mismatch — restart the session and Run all again."
import src.pipeline.b0 as b0
%cd /content
if not Path("/content/sam2_repo/sam2/__init__.py").exists():
    !git clone -q https://github.com/facebookresearch/sam2.git /content/sam2_repo
    !git -C /content/sam2_repo checkout -q {b0.FROZEN_SAM2_COMMIT}
    %cd /content/sam2_repo
    !pip install -q -e .
    %cd /content
if "/content/sam2_repo" not in sys.path:
    sys.path.insert(0, "/content/sam2_repo")
importlib.invalidate_caches()

import src.datasets as datasets_mod
import src.pipeline.semantic as semantic
import src.pipeline.prompts as prompts
import src.pipeline.candidates as C1
import src.pipeline.candidates_v2 as core

SAN_ROOT = Path("/content/SAN"); SAN_CKPT = SAN_ROOT / "output" / "san_vit_b_16.pth"
SAN_CKPT.parent.mkdir(parents=True, exist_ok=True)
if not SAN_CKPT.exists():
    shutil.copy2(PROJECT_ROOT / "Checkpoints" / "SAN" / "san_vit_b_16.pth", SAN_CKPT)
SAM2_CKPT = PROJECT_ROOT / "Checkpoints" / "SAM2" / "sam2.1_hiera_large.pt"
M = core.load_models(SAN_ROOT, SAN_CKPT, SAM2_CKPT, systems=("A", "B"), device="cuda")
# ---- Frozen-code check: the pipeline files are unchanged since the B5.1 freeze
FREEZE = "2086087c5e"
FROZEN_FILES = ["src/pipeline/candidates_v2.py", "src/pipeline/candidates.py", "src/pipeline/gpv.py",
                "src/pipeline/prompts.py", "src/pipeline/semantic.py", "src/pipeline/b0.py"]
r = subprocess.run(["git", "-C", str(REPO_DIR), "diff", "--quiet", FREEZE, "HEAD", "--"] + FROZEN_FILES)
assert r.returncode == 0, "Pipeline code differs from the B5.1 freeze — stop."
RULE = json.load(open(RESULTS_ROOT / "B6_0_Setup_and_Rules" / "B6_rule_file.json"))
print("Frozen code unchanged since", FREEZE, "| rule file recorded", RULE["recorded_utc"])
print(core.VERSION, "| models loaded:", "SAN, SAM2 predictor,", sorted(M.amg), "| normals:", M.normals is not None)
from tqdm.auto import tqdm
from PIL import Image

import src.evaluation as ev

/content
/content/sam2_repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for SAM-2 (pyproject.toml) ... done
/content


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
100%|███████████████████████████████████████| 351M/351M [00:13<00:00, 26.9MiB/s]
/content/SAN/san/model/side_adapter/timm_wrapper.py:24: UserWarning: Unused kwargs are provided:{'dynamic_img_pad': False, 'device': None, 'dtype': None}.
  warnings.warn(f"Unused kwargs are provided:{kwargs}.")


model_index.json:   0%|          | 0.00/512 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

Frozen code unchanged since 2086087c5e | rule file recorded 2026-10-26T13:32:37+00:00
candidates_v2 | models loaded: SAN, SAM2 predictor, ['LM0', 'MixA'] | normals: True


In [4]:
# ============================================================
# Step 0.4 — Test items: OmniCount-191 test (Drive) and VOC 2007 test (extracted locally)
# ============================================================
import tarfile
om = datasets_mod.OmniCount191Dataset(OMNICOUNT_ROOT, split="test", load_images=False)
VOCAB_TEST = om.domain_vocabulary
ITEMS = {"omnicount": [], "voc": []}
for i in range(len(om)):
    s = om[i]; d = s["domain"]
    ITEMS["omnicount"].append({"key": s["image_id"].replace("/", "__"), "image_path": s["image_path"], "classes": VOCAB_TEST[d],
                               "prompts": {c: datasets_mod.omnicount_prompt_name(c) for c in VOCAB_TEST[d]},
                               "meta": {"dataset": "omnicount", "domain": d, "idx": i, "image_id": s["image_id"],
                                        "gt": {c: v["count"] for c, v in s["categories"].items()}}})
VOC_LOCAL = LOCAL / "voc2007"
if not (VOC_LOCAL / "VOCdevkit" / "VOC2007" / "ImageSets").exists():
    with tarfile.open(VOC_TAR) as t:
        t.extractall(VOC_LOCAL, filter="data")
voc = datasets_mod.VOC2007CountingDataset(VOC_LOCAL, "test", include_difficult=True, load_images=False)
for i in range(len(voc)):
    s = voc[i]
    ITEMS["voc"].append({"key": Path(s["image_path"]).stem, "image_path": s["image_path"], "classes": list(datasets_mod.VOC_CLASSES),
                         "prompts": dict(s["prompt_names"]),
                         "meta": {"dataset": "voc2007", "domain": "VOC2007", "image_id": s["image_id"],
                                  "gt_all": {c: v["count_all"] for c, v in s["categories"].items()},
                                  "gt_nondifficult": {c: v["count_nondifficult"] for c, v in s["categories"].items() if v["count_nondifficult"] > 0}}})
inv = RULE["inventory"]
assert len(ITEMS["omnicount"]) == inv["omnicount_test"]["images"] == 1957
assert len(ITEMS["voc"]) == inv["voc"]["images"] == 4952
assert sum(sum(it["meta"]["gt_all"].values()) for it in ITEMS["voc"]) == inv["voc"]["objects_all"]
assert sum(len(it["classes"]) for it in ITEMS["omnicount"]) == inv["omnicount_test"]["standard_queries"]
print({k: len(v) for k, v in ITEMS.items()}, "| OmniCount standard queries:", inv["omnicount_test"]["standard_queries"],
      "| VOC objects (all):", inv["voc"]["objects_all"])

{'omnicount': 1957, 'voc': 4952} | OmniCount standard queries: 31645 | VOC objects (all): 14976


In [5]:
# ============================================================
# Step 1.1 — Sanity check: 3 OmniCount val held-out images from B5.2 (L4) must reproduce
# ============================================================
B52 = RESULTS_ROOT / "B5_2_PreTest_Validation" / "omnicount_val_heldout" / "rows"
val_ds = datasets_mod.OmniCount191Dataset(OMNICOUNT_ROOT, split="val", load_images=False)
cands = []
for p in sorted(B52.glob("*.json"))[:60]:
    r = json.load(open(p))
    if sum(r["counts"]["A"].values()) > 0:
        cands.append(r)
rows = []
for r in [cands[0], cands[len(cands) // 2], cands[-1]]:
    d = r["meta"]["domain"]; classes = val_ds.domain_vocabulary[d]
    out = core.count_image(M, r["image_path"], classes, "multi", systems=("A", "B"),
                           prompt_of={c: datasets_mod.omnicount_prompt_name(c) for c in classes})
    for s_ in "AB":
        diffs = [abs(out["counts"][s_][c] - r["counts"][s_][c]) for c in classes]
        rows.append({"image_id": r["meta"]["image_id"], "system": s_, "classes": len(classes),
                     "sum live": sum(out["counts"][s_].values()), "sum B5.2": sum(r["counts"][s_].values()),
                     "classes differing": int(sum(x > 0 for x in diffs)), "max |diff|": max(diffs),
                     "t (s)": round(out["diag"]["t_total_s"], 1)})
san = pd.DataFrame(rows); display(san)
dev = int(san["max |diff|"].max())
assert dev <= 2, "Live counts differ from B5.2 by more than 2 — stop and check the environment."
print("SANITY CHECK: PASS" + (" (exact)" if dev == 0 else " (small GPU differences)"))

,image_id,system,classes,sum live,sum B5.2,classes differing,max |diff|,t (s)
0,Fruits/val/101,A,8,6,6,0,0,25.0
1,Fruits/val/101,B,8,6,6,0,0,25.0
2,Fruits/val/199,A,8,6,6,0,0,22.0
3,Fruits/val/199,B,8,6,6,0,0,22.0
4,Fruits/val/32,A,8,7,7,0,0,23.5
5,Fruits/val/32,B,8,7,7,0,0,23.5


SANITY CHECK: PASS (exact)


In [6]:
# ============================================================
# Step 2.1 — Our two systems, multi-label mode (U_both), every class queried (resumable)
# ============================================================
RUN = ["omnicount", "voc"]                     # e.g. ["omnicount"] to run one dataset this session
for name in RUN:
    t0 = time.time()
    n = core.run_batch(M, ITEMS[name], OUT_DIR / name / "rows_ours", "multi", systems=("A", "B"))
    print(f"{name}: processed now {n} | {(time.time() - t0) / 3600:.2f} h")

candidates_v2 multi: 0it [00:00, ?it/s]

omnicount: processed now 0 | 0.00 h


candidates_v2 multi:   0%|          | 0/3014 [00:00<?, ?it/s]

voc: processed now 3014 | 17.35 h


In [7]:
# ============================================================
# Step 3.1 — Completeness, progress summary (descriptive; full analysis in B6_5), metadata
# ============================================================
SYS = {"A": "GPV_R4+P[MixA]", "B": "GPV_K-n+P[LM0]"}
def load_rows(d):
    return {p.stem: json.load(open(p)) for p in Path(d).glob("*.json")}
def summarise(results, omnicount_metric):
    _, _, m = ev.evaluate_multi_label_counts(ev.build_multi_label_pairs(results))
    row = {"standard mRMSE": m["mRMSE"], "mRMSE-nz": m["mRMSE_nz"], "MAE_nz": m["MAE_nz"], "RMSE_nz": m["RMSE_nz"],
           "NAE_nz": m["NAE_nz"], "FP units": m["false_positive_units"]}
    if omnicount_metric:
        _, om_ = ev.omnicount_reference_by_domain(results); row["OmniCount m_rmse_nz"] = om_["pooled_m_rmse_nz"]
    return row
summary, status = [], {}
for name, gt_keys in [("omnicount", ["gt"]), ("voc", ["gt_all", "gt_nondifficult"])]:
    rows = load_rows(OUT_DIR / name / "rows_ours")
    keys = [it["key"] for it in ITEMS[name]]
    status[name] = f"{sum(k in rows for k in keys)}/{len(keys)}"
    if not all(k in rows for k in keys):
        print(f"{name}: incomplete ({status[name]}) — Run all again to continue"); continue
    for gk in gt_keys:
        for s_ in "AB":
            res = [{"image_id": rows[k]["meta"]["image_id"], "domain": rows[k]["meta"]["domain"],
                    "categories": {c: {"count": n} for c, n in rows[k]["meta"][gk].items()},
                    "pred_counts": rows[k]["counts"][s_], "query_classes": rows[k]["classes"]} for k in keys]
            summary.append({"dataset": name, "gt": gk, "system": SYS[s_], **summarise(res, name == "omnicount")})
    print(f"{name}: mean time per image (both systems) {np.mean([rows[k]['diag']['t_total_s'] for k in keys]):.1f} s")
if summary:
    sm = pd.DataFrame(summary)
    print("Progress summary — standard protocol, all classes queried (descriptive; B6_5 runs the pre-declared analysis):")
    display(sm.round(3)); sm.to_csv(OUT_DIR / "B6_2_progress_summary.csv", index=False)
done = all(v.split("/")[0] == v.split("/")[1] for v in status.values())
with open(OUT_DIR / "B6_2_metadata.json", "w") as f:
    json.dump({"notebook": "B6_2_Ours_Multi", "module": core.VERSION, "freeze_commit": FREEZE, "repo_commit": REPO_COMMIT,
               "gpu": GPU_NAME, "rule_file_recorded_utc": RULE["recorded_utc"], "status": status, "complete": bool(done),
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "torch": torch.__version__}, f, indent=2)
print("COMPLETE" if done else "NOT COMPLETE", status, "| saved to", OUT_DIR)

omnicount: mean time per image (both systems) 22.8 s
voc: mean time per image (both systems) 20.9 s
Progress summary — standard protocol, all classes queried (descriptive; B6_5 runs the pre-declared analysis):


,dataset,gt,system,standard mRMSE,mRMSE-nz,MAE_nz,RMSE_nz,NAE_nz,FP units,OmniCount m_rmse_nz
0,omnicount,gt,GPV_R4+P[MixA],2.239,7.308,2.849,8.674,1.155,10077.0,0.359
1,omnicount,gt,GPV_K-n+P[LM0],2.679,8.787,3.359,10.425,1.362,9668.0,0.433
2,voc,gt_all,GPV_R4+P[MixA],2.234,8.198,4.912,8.430,3.753,14405.0,NaN
3,voc,gt_all,GPV_K-n+P[LM0],4.599,15.851,6.936,15.947,5.322,19994.0,NaN
4,voc,gt_nondifficult,GPV_R4+P[MixA],2.299,8.724,5.223,8.793,4.121,17090.0,NaN
5,voc,gt_nondifficult,GPV_K-n+P[LM0],4.646,16.379,7.373,16.481,5.853,22623.0,NaN


COMPLETE {'omnicount': '1957/1957', 'voc': '4952/4952'} | saved to /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B6_2_Ours_Multi
